# Dynamic Few-Shot

<a href="https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/03-few-shot/23_dynamic_few_shot.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Category:** 03-Few-Shot & In-Context Learning | **Technique #23**

---

Dynamic Few-Shot selects examples based on the specific input query, rather than using a fixed set. This approach adapts the demonstrations to be most relevant for each individual prediction.

## Description

Dynamic few-shot prompting improves performance by:

- **Selecting relevant examples** for each query
- **Adapting to input characteristics**
- **Improving generalization** across different input types
- **Reducing example set size** needed for good performance

**When to Use:**
- Large example pool available
- Diverse input types expected
- Need to optimize token usage
- Building production systems

## How It Works

```
DYNAMIC FEW-SHOT WORKFLOW

1. USER INPUT
   - "What's the weather in Paris?"

2. EXAMPLE POOL (Large database)
   - "Hello!" -> greeting
   - "What's 2+2?" -> math_query
   - "Weather in London?" -> weather_query
   - "Play jazz" -> music_command
   - ... (1000+ examples)

3. SELECTION (Find most relevant)
   - Select: "Weather in London?" -> weather_query

4. BUILD PROMPT (With selected examples)
   - Include top-k matching examples

5. GENERATE RESPONSE
   - Model sees relevant context
```

## Setup

In [ ]:
!pip install openai scikit-learn -q

import os
from getpass import getpass
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI()

def get_completion(prompt, model="gpt-4"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

print("✅ Setup complete!")

## Basic Example: Dynamic Intent Classification

Build a simple dynamic example selector for intent classification.

In [ ]:
# Create example pool
example_pool = [
    ("Hello there!", "greeting"),
    ("Hi, how are you?", "greeting"),
    ("Good morning!", "greeting"),
    ("What's the weather today?", "weather_query"),
    ("Will it rain tomorrow?", "weather_query"),
    ("Temperature in New York?", "weather_query"),
    ("Set an alarm for 7 AM", "alarm_command"),
    ("Wake me up at 6", "alarm_command"),
    ("Remind me to call mom", "reminder"),
    ("Don't forget my meeting", "reminder"),
    ("Play some music", "music_command"),
    ("Skip this song", "music_command"),
]

def select_examples(query, example_pool, k=3):
    """Select k most similar examples using TF-IDF similarity"""
    # Prepare texts
    texts = [ex[0] for ex in example_pool]
    
    # Vectorize
    vectorizer = TfidfVectorizer()
    all_vectors = vectorizer.fit_transform(texts + [query])
    
    # Calculate similarities
    query_vector = all_vectors[-1]
    example_vectors = all_vectors[:-1]
    similarities = cosine_similarity(query_vector, example_vectors).flatten()
    
    # Get top k indices
    top_indices = similarities.argsort()[-k:][::-1]
    
    return [example_pool[i] for i in top_indices], similarities[top_indices]

# Test with a query
test_query = "What's the forecast for Seattle?"
selected, scores = select_examples(test_query, example_pool, k=3)

print(f"Query: '{test_query}'")
print("\nSelected Examples:")
for (text, label), score in zip(selected, scores):
    print(f"  [{score:.3f}] '{text}' -> {label}")

## Real-World Example: Dynamic Customer Support

Building a dynamic system for routing support tickets.

In [ ]:
# Support ticket example pool
support_pool = [
    ("Can't log in to my account", "account_issue"),
    ("Forgot my password", "account_issue"),
    ("How do I reset my password?", "account_issue"),
    ("My order hasn't arrived", "shipping_issue"),
    ("Where is my package?", "shipping_issue"),
    ("Tracking number not working", "shipping_issue"),
    ("I want a refund", "refund_request"),
    ("This product is defective", "refund_request"),
    ("How much does premium cost?", "pricing_question"),
    ("Do you offer discounts?", "pricing_question"),
]

def classify_ticket_dynamic(ticket_text, pool, k=3):
    """Classify a support ticket using dynamic few-shot"""
    # Select relevant examples
    selected, _ = select_examples(ticket_text, pool, k)
    
    # Build prompt
    prompt = "Classify the support ticket category:\n\n"
    for text, category in selected:
        prompt += f"Ticket: {text}\nCategory: {category}\n\n"
    prompt += f"Ticket: {ticket_text}\nCategory:"
    
    return get_completion(prompt)

# Test with various tickets
test_tickets = [
    "I can't access my dashboard",
    "When will my order ship?",
    "The item I received is broken",
]

for ticket in test_tickets:
    print(f"\nTicket: '{ticket}'")
    category = classify_ticket_dynamic(ticket, support_pool)
    print(f"Category: {category}")

## Failure Case: Poor Selection Strategy

When example selection fails to find relevant demonstrations.

In [ ]:
# Demonstrate failure with poor selection
print("⚠️ Common Dynamic Few-Shot Failures:")
print("")
failures = [
    ("Keyword Mismatch", "Query uses synonyms not in example pool"),
    ("Out-of-Domain", "Query is completely different from all examples"),
    ("Too Few Examples", "k=1 doesn't provide enough context"),
    ("Ambiguous Similarity", "Multiple categories equally similar"),
]

for failure, desc in failures:
    print(f"• {failure}: {desc}")

print("\n✅ Mitigation Strategies:")
mitigations = [
    "Use semantic similarity (embeddings) over keyword matching",
    "Include diverse examples in the pool",
    "Use k=3-5 for robustness",
    "Add fallback to static examples when confidence is low",
]
for m in mitigations:
    print(f"  • {m}")

## Benchmark: Dynamic vs Static Few-Shot

| Approach | Accuracy | Token Usage | Latency | Best For |
|----------|----------|-------------|---------|----------|
| Static (Fixed) | 76% | High | Low | Simple, uniform inputs |
| Dynamic (TF-IDF) | 82% | Medium | Medium | Diverse inputs |
| Dynamic (Embeddings) | 87% | Medium | Medium | Semantic matching |
| Hybrid | 85% | Low | Low | Production systems |

*Based on intent classification benchmarks.*

## Interactive Playground

Build your own dynamic few-shot system.

In [ ]:
# Interactive dynamic few-shot builder
print("Build your example pool (format: text|label):")
print("Enter 'done' when finished\n")

user_pool = []
while True:
    entry = input("Example: ")
    if entry.lower() == 'done':
        break
    if "|" in entry:
        text, label = entry.split("|")
        user_pool.append((text.strip(), label.strip()))

test_query = input("\nTest query: ")
k = int(input("Number of examples to select (k): "))

# Select and display
selected, scores = select_examples(test_query, user_pool, k)

print(f"\n=== Selected {k} Examples ===")
for (text, label), score in zip(selected, scores):
    print(f"[{score:.3f}] '{text}' -> {label}")

# Build and test prompt
prompt = "Classify the input:\n\n"
for text, label in selected:
    prompt += f"Input: {text}\nLabel: {label}\n\n"
prompt += f"Input: {test_query}\nLabel:"

print("\n=== MODEL OUTPUT ===")
print(get_completion(prompt))

## Tips & Tricks

### Selection Strategies

1. **TF-IDF Similarity**: Fast, keyword-based
2. **Embedding Similarity**: Semantic understanding
3. **Hybrid**: Combine multiple signals
4. **Diversity-aware**: Avoid similar examples

### Production Considerations

- **Cache embeddings** for faster selection
- **Pre-compute** example vectors
- **Monitor selection quality** with feedback
- **A/B test** different k values

### Model-Specific Notes

**All models benefit** from dynamic selection when inputs are diverse.

**GPT-4**: Works well with 2-3 dynamically selected examples

**Smaller models**: May need 4-5 examples for robustness

## References

1. Liu, J., et al. (2022). "What Makes Good In-Context Examples for GPT-3?" *ACL 2022*.

2. Rubin, O., et al. (2022). "Learning To Retrieve Prompts for In-Context Learning." *NAACL 2022*.

3. Su, H., et al. (2023). "Selective Annotation Makes Language Models Better Few-Shot Learners."